In [ ]:
# This notebook was opened and run on colab, with dataset stored on drive. If already in the repo just skip the setup and run the training loop and adjust file paths
!git clone https://github.com/aaravd18/transformer-chess.git
%cd transformer-chess
!pip install -r requirements.txt

In [ ]:
import wandb
wandb.login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from data.preprocess import copy_drive_to_local
DRIVE_DIR = "/content/drive/MyDrive/Research/chess-transformer/"

copy_drive_to_local(DRIVE_DIR + "data", "datasets")

In [ ]:
"""
Training loop for the chess transformer.
For an overfit sanity test set MAX_STEPS=500, BATCH_SIZE=32, WARMUP_STEPS=50, LR=1e-3
"""
import math
import wandb
import time
import torch
from model import *
from data.dataloader import make_loader

torch.set_float32_matmul_precision("high")

DATA_DIR = "datasets/"

# ---- Logging / wandb ----
LOG_EVERY     = 100  # also how often val eval runs
VAL_MAX_BATCHES  = 10
WANDB_PROJECT = "chess-transformer"
WANDB_RUN_NAME = None  # wandb autogenerates

# ---- Training hyperparameters ----
BATCH_SIZE    = 2048
LR            = 6.5e-4
WEIGHT_DECAY  = 0.1
WARMUP_STEPS  = 500
MAX_STEPS     = 7000
NUM_WORKERS   = 2
GRAD_CLIP     = 1.0


# ---- Checkpoint paths ----
CKPT_BEST     = DRIVE_DIR + "ckpt_best.pt"
CKPT_FINAL    = DRIVE_DIR + "ckpt_final.pt"

# ---- Reproducibility ----
SEED          = 0



def get_state_dict(model):
    """Get state dict from the underlying model, unwrapping torch.compile."""
    return getattr(model, "_orig_mod", model).state_dict()


# ---------------------------------------------------------------------------
# LR schedule: linear warmup -> cosine decay to 10% of peak
# ---------------------------------------------------------------------------

def make_lr_lambda(warmup_steps: int, total_steps: int):
    """Linear warmup, then cosine decay to 10% of peak LR.

    Returns a function step -> multiplier in [0, 1] that AdamW's base LR
    gets multiplied by. We never decay to zero -- the 10% floor avoids
    dying-LR pathologies near the end of training.
    """
    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        progress = min(1.0, progress)
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))
    return lr_lambda


# ---------------------------------------------------------------------------
# Validation eval -- average loss + top-1 accuracy over the whole val set
# ---------------------------------------------------------------------------

@torch.no_grad()
def evaluate(model, val_loader, device, max_batches: int | None = None) -> tuple[float, float]:
    """Mean val loss and top-1 (from, to) accuracy.

    If `max_batches` is set, evaluate on at most that many batches.
    Caller is responsible for flipping back to model.train() afterward.
    """
    model.eval()
    total_loss     = 0.0
    total_correct  = 0
    total_examples = 0

    use_bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()

    for i, batch in enumerate(val_loader):
        if max_batches is not None and i >= max_batches:
            break

        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
            out = model(batch["tokens"])
            loss = policy_loss(
                out["move_logits"], out["promo_logits"],
                batch["from_sq"], batch["to_sq"], batch["promotion"],
            )

        B = out["move_logits"].shape[0]
        flat   = out["move_logits"].reshape(B, 64 * 64)
        pred   = flat.argmax(dim=-1)
        target = batch["from_sq"] * 64 + batch["to_sq"]

        total_correct  += (pred == target).sum().item()
        total_examples += B
        total_loss     += loss.item() * B

    return total_loss / total_examples, total_correct / total_examples


# ---------------------------------------------------------------------------
# Optimizer construction with weight-decay split
# ---------------------------------------------------------------------------

def build_optimizer(model: nn.Module, lr: float, weight_decay: float):
    """AdamW with weight decay only on weight matrices, not biases / LN.

    Standard transformer convention. Picks up Embedding weights too,
    which is the desired behaviour -- you generally do want to decay
    those.
    """
    decay_params, no_decay_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2:               # weight matrices, embeddings
            decay_params.append(p)
        else:                         # biases, LayerNorm gains
            no_decay_params.append(p)

    return torch.optim.AdamW(
        [
            {"params": decay_params,    "weight_decay": weight_decay},
            {"params": no_decay_params, "weight_decay": 0.0},
        ],
        lr=lr,
        # 0.95 (vs default 0.999) is more stable for transformers; this
        # is the choice GPT-3, Chinchilla, etc. use.
        betas=(0.9, 0.95),
    )


# ---------------------------------------------------------------------------
# Main training loop
# ---------------------------------------------------------------------------

def train():
    # ---- Reproducibility ----
    torch.manual_seed(SEED)

    # ---- Device ----
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Device: {device}")

    USE_BF16 = device.type == "cuda" and torch.cuda.is_bf16_supported()
    print("BF16 supported:", USE_BF16)

    cfg = ModelConfig()
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        entity="aaravdesai-university-of-california-berkeley",
        config={
            # Model
            **cfg.__dict__,
            # Training
            "batch_size":    BATCH_SIZE,
            "lr":            LR,
            "weight_decay":  WEIGHT_DECAY,
            "warmup_steps":  WARMUP_STEPS,
            "max_steps":     MAX_STEPS,
            "grad_clip":     GRAD_CLIP,
            "seed":          SEED,
            "device":        str(device),
        },
    )

    # ---- Data ----
    train_loader = make_loader(
        data_dir=DATA_DIR,
        split='train',
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
    )

    val_loader = make_loader(
        data_dir=DATA_DIR,
        split='val',
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
    )


    print(f"Train batches/epoch: {len(train_loader):,}   "
          f"Val batches: {len(val_loader):,}")

    # ---- Model ----
    model = ChessTransformer(cfg).to(device)
    print(f"Model: {model.num_params():,} parameters")

    if device.type == "cuda":
        model = torch.compile(model)

    # ---- Optimizer + scheduler ----
    optimizer = build_optimizer(model, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, make_lr_lambda(WARMUP_STEPS, MAX_STEPS)
    )

    # ---- Logging state ----
    t0 = time.time()
    best_val_loss = float("inf")

    print(f"\n{'step':>7}  {'train_loss':>10}  {'lr':>9}  "
          f"{'val_loss':>9}  {'val_acc':>8}  {'elapsed':>9}")
    print("-" * 72)

    # ---- Loop ----
    # Step-based (not epoch-based) so MAX_STEPS works cleanly regardless
    # of dataset size. The train iterator is recycled when exhausted --
    # equivalent to walking multiple epochs.
    step = 0
    train_iter = iter(train_loader)
    model.train()

    while step < MAX_STEPS:
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
            out = model(batch["tokens"])
            loss = policy_loss(
                out["move_logits"], out["promo_logits"],
                batch["from_sq"], batch["to_sq"], batch["promotion"],
            )

        optimizer.zero_grad()
        loss.backward()
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        step += 1

        # ---- Periodic logging + val eval ----
        if step % LOG_EVERY == 0 or step == 1:
            current_lr = scheduler.get_last_lr()[0]
            val_loss, val_acc = evaluate(model, val_loader, device, max_batches=VAL_MAX_BATCHES)
            elapsed = time.time() - t0
            print(f"{step:>7}  {loss.item():>10.4f}  {current_lr:>9.2e}  "
                  f"{val_loss:>9.4f}  {val_acc:>7.1%}  {elapsed:>8.1f}s")

            wandb.log({
                "train/loss":       loss.item(),
                "train/grad_norm":  grad_norm.item(),
                "train/lr":         current_lr,
                "val/loss":         val_loss,
                "val/accuracy":     val_acc,
                "elapsed_seconds":  elapsed,
            }, step=step)

            # Save best-by-val-loss separately from final.
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save({
                    "step":      step,
                    "model":     get_state_dict(model),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "cfg":       cfg,
                    "val_loss":  val_loss,
                    "val_acc":   val_acc,
                }, CKPT_BEST)

            # evaluate() flipped us to eval mode; flip back.
            model.train()

    # ---- Final checkpoint (always saved, regardless of val) ----
    torch.save({
        "step":      step,
        "model":     get_state_dict(model),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "cfg":       cfg,
    }, CKPT_FINAL)

    elapsed = time.time() - t0
    print(f"\nDone in {elapsed:.1f}s.")
    print(f"  Best val loss: {best_val_loss:.4f}  ({CKPT_BEST})")
    print(f"  Final ckpt:    {CKPT_FINAL}")

    wandb.summary["best_val_loss"] = best_val_loss
    wandb.summary["total_seconds"] = elapsed

    wandb.finish()


if __name__ == "__main__":
    train()